# Expert-CFG: VQA-RAD 평가 실행

**필요한 Kaggle 데이터셋:**
```
/kaggle/input/expert-cfg-data/
├── VQA-RAD/images/   # VQA-RAD 이미지
└── test.json         # build_test_json.ipynb로 생성한 파일

/kaggle/input/datasets/ciderchicken/expert-cfg-weights/
└── Phi-3.5-vision-instruct/   # 모델 가중치
```

## Step 1. 패키지 설치

In [ ]:
!pip install transformers==4.43.0 accelerate==0.30.0 peft==0.11.1 Pillow==10.3.0 -q

## Step 2. Expert-CFG 저장소 클론

In [ ]:
import subprocess

if not os.path.exists("/kaggle/working/Expert-CFG"):
    subprocess.run([
        "git", "clone",
        "https://github.com/minseongkim-dev/expert-cfg",
        "/kaggle/working/Expert-CFG"
    ], check=True)
    print("클론 완료")
else:
    print("이미 존재")

## Step 3. Phi-3.5-vision-instruct 모델 다운로드 (~10분)

## Step 4. 경로 확인

## Step 5. demo.py 실행 (~1-2시간)

- `--cfg 1.5` : CFG guidance scale γ
- `--attn 3.0` : attention highlight β
- `--use_attn_patch` : Expert Highlight 활성화
- `--bf16` : bfloat16 (T4에서 메모리 절약)

> OOM 발생 시 `--use_qlora` 추가

In [ ]:
import sys

cmd = [
    sys.executable, "demo.py",
    "--dataset",            "vqa-rad",
    "--input_json",         TEST_JSON,
    "--img_root",           IMG_ROOT,
    "--model_name_or_path", MODEL_DIR,
    "--cfg",                "1.5",
    "--attn",               "3.0",
    "--use_attn_patch",
    "--bf16",
    "--save_path",          SAVE_PATH,
]

print("실행 명령:", " ".join(cmd))
result = subprocess.run(cmd, cwd="/kaggle/working/Expert-CFG")
print(f"\n종료 코드: {result.returncode}")

import os

MODEL_DIR  = "/kaggle/input/datasets/ciderchicken/expert-cfg-weights/Phi-3.5-vision-instruct"
DATA_ROOT  = "/kaggle/input/expert-cfg-data"
TEST_JSON  = f"{DATA_ROOT}/test.json"
IMG_ROOT   = f"{DATA_ROOT}/VQA-RAD/images"
SAVE_PATH  = "/kaggle/working/vqa-rad-results.json"

for label, path in [("model", MODEL_DIR), ("test.json", TEST_JSON), ("images/", IMG_ROOT)]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}")

if os.path.exists(IMG_ROOT):
    imgs = [f for f in os.listdir(IMG_ROOT) if f.endswith(".jpg")]
    print(f"   이미지 수: {len(imgs)}개")

In [ ]:
import json

with open(SAVE_PATH) as f:
    output = json.load(f)

meta    = output["meta"]
summary = output["summary"]
results = output["results"]

print(f"=== 결과 요약 ===")
print(f"데이터셋  : {meta['dataset']}")
print(f"모델      : {meta['model']}")
print(f"CFG γ     : {meta['cfg']}")
print(f"Attn β    : {meta['attn_beta']}")
print(f"정확도    : {summary['correct']}/{summary['total']} = {summary['accuracy']:.4f}")

print(f"\n=== 샘플 5개 ===")
for r in results[:5]:
    mark = "✓" if r["correct"] else "✗"
    print(f"[{mark}] Q: {r['question'][:50]}")
    print(f"     GT: {r['answer']}  Pred: {r['prediction']}")